# Week 8: NLP Foundations & Text Embeddings
## Notebook 07: FastText Subword Embeddings & OOV Generalization

### 1. Conceptual Intuition & Theoretical Background
Traditional Word2Vec models assign a distinct embedding vector to each atomic word in a fixed vocabulary. When encountering an Out-of-Vocabulary (OOV) word or typographical misspelling (e.g., "artificialll"), Word2Vec fails and outputs an unknown token `[UNK]`.

**FastText** represents each word as a bag of **character n-grams** bounded by special delimiter tags `< ` and ` >`:
For word `<where>` with $n=3$:
$$\mathcal{G}_{\text{where}} = \{ \text{<wh},\ \text{whe},\ \text{her},\ \text{ere},\ \text{re>},\ \text{<where>} \}$$

The final vector representation $v_w$ is the sum of its subword n-gram embeddings:
$$v_w = \sum_{g \in \mathcal{G}_w} z_g$$

To bound the memory of infinite potential character n-grams, FastText maps n-grams into a fixed-size embedding table (e.g., 2,000,000 buckets) using the **Fowler-Noll-Vo (FNV-1a)** hash trick.

---

### 2. Scratch Python Implementation: FastText Subword Generator

In [ ]:
import numpy as np

class FastTextScratch:
    def __init__(self, embedding_dim=16, num_buckets=5000, min_n=3, max_n=5):
        self.dim = embedding_dim
        self.num_buckets = num_buckets
        self.min_n = min_n
        self.max_n = max_n
        # Subword embedding table
        np.random.seed(42)
        self.subword_embeddings = np.random.normal(0, 0.1, (num_buckets, embedding_dim))
        
    def _hash(self, ngram):
        """Fowler-Noll-Vo style deterministic hashing to bucket index."""
        h = 2166136261
        for char in ngram:
            h = (h ^ ord(char)) * 16777619
        return (h & 0x7FFFFFFF) % self.num_buckets
        
    def get_ngrams(self, word):
        tagged = f"<{word}>"
        ngrams = []
        # Extract character n-grams from min_n to max_n
        for n in range(self.min_n, self.max_n + 1):
            for i in range(len(tagged) - n + 1):
                ngrams.append(tagged[i : i + n])
        ngrams.append(tagged)  # Include full tagged token
        return ngrams
        
    def get_word_vector(self, word):
        ngrams = self.get_ngrams(word)
        bucket_indices = [self._hash(g) for g in ngrams]
        # Sum of subword embeddings
        vec = np.sum(self.subword_embeddings[bucket_indices], axis=0)
        norm = np.linalg.norm(vec)
        return vec / (norm + 1e-9)

# Instantiate FastText Model
ft = FastTextScratch(embedding_dim=8, num_buckets=2000, min_n=3, max_n=4)

# Test Out-of-Vocabulary (OOV) Robustness
v_correct = ft.get_word_vector("neural")
v_typo = ft.get_word_vector("neuural")  # Misspelled
v_unrelated = ft.get_word_vector("banana")

# Cosine similarity
def cosine_sim(a, b):
    return np.dot(a, b)

sim_typo = cosine_sim(v_correct, v_typo)
sim_diff = cosine_sim(v_correct, v_unrelated)

print("--- FastText Subword Compositionality ---")
print(f"Cosine Similarity ('neural' vs 'neuural' [Typo]): {sim_typo:.4f}")
print(f"Cosine Similarity ('neural' vs 'banana'):         {sim_diff:.4f}")
print("FastText successfully constructs meaningful vector representations for OOV misspellings!")

### 3. Step-by-Step Code Walkthrough
1. **`get_ngrams`**: Deconstructs words into character prefixes, infixes, and suffixes (e.g. `<wh`, `ere>`).
2. **Deterministic Hash Hashing**: Maps arbitrary strings into bounded table buckets $0 \dots B-1$.
3. **Subword Aggregation**: Typographical variations share majority character n-grams, yielding high cosine similarity.

---

### 4. Industry Applications & Trade-offs
- **Morphologically Rich Languages (German, Turkish, Urdu):** Efficiently represents compound and inflected words without vocabulary explosion.
- **Trade-off:** Pre-computing word embeddings requires summing multiple n-grams, increasing memory bandwidth during inference.

---

### 5. Top Interview Questions & Answers
**Q1: How does FastText solve the Out-of-Vocabulary (OOV) problem?**
> **Answer:** By representing each word as the sum of its character n-gram embeddings, an unseen word can still be synthesized by summing its constituent subword n-grams.

**Q2: What is the purpose of the hashing trick in FastText?**
> **Answer:** The number of unique character 3-grams to 6-grams is enormous. Hashing maps n-grams into a fixed number of buckets (e.g., $2 \times 10^6$), capping memory usage at the cost of rare hash collisions.